# WavLM Embeddings + Text XGBoost (Weighted Vote)

Two independent models combined via weighted voting:
1. **WavLM XGBoost** — audio embeddings (768-dim)
2. **Text XGBoost** — original 41 text+pause+prosodic features

No wav2vec2.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
import xgboost as xgb
import joblib

FEATURES_CSV = "features_company.csv"
WAVLM_CSV = "features_wavlm.csv"

print(f"Text features:  {FEATURES_CSV}")
print(f"WavLM features: {WAVLM_CSV}")

## 1. Extract WavLM Embeddings (CPU)

In [ ]:
!pip install -q transformers torch librosa soundfile tqdm

In [ ]:
import torch
import librosa
from tqdm import tqdm
from transformers import AutoFeatureExtractor, WavLMModel

df = pd.read_csv(FEATURES_CSV)
df = df[df["label_int"].isin([0, 1])].reset_index(drop=True)
print(f"Loaded {len(df)} labelled samples")

print("Loading WavLM-base-plus...")
feature_extractor = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
wavlm = WavLMModel.from_pretrained("microsoft/wavlm-base-plus")
wavlm = wavlm.eval().to("cpu")

EMBED_DIM = wavlm.config.hidden_size
print(f"Embedding dim: {EMBED_DIM}")

In [ ]:
SR = 16000
MAX_DURATION = 60  # use full 60s of audio

embeddings = []
failed = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting WavLM embeddings"):
    fp = row.get("filepath", "")
    if not fp or not os.path.exists(fp):
        embeddings.append(np.zeros(EMBED_DIM))
        failed.append(idx)
        continue
    try:
        audio, _ = librosa.load(fp, sr=SR, mono=True, duration=MAX_DURATION)
        if len(audio) < SR:
            embeddings.append(np.zeros(EMBED_DIM))
            failed.append(idx)
            continue

        with torch.no_grad():
            inputs = feature_extractor(
                audio, sampling_rate=SR, return_tensors="pt", padding=True
            )
            outputs = wavlm(**inputs)
            hidden = outputs.last_hidden_state  # (1, T, 768)
            emb = hidden[0].mean(dim=0).numpy()

        embeddings.append(emb)
    except Exception as e:
        print(f"  Failed {fp}: {e}")
        embeddings.append(np.zeros(EMBED_DIM))
        failed.append(idx)

print(f"\nExtracted {len(embeddings)} embeddings, {len(failed)} failed")

embed_cols = [f"wavlm_{i}" for i in range(EMBED_DIM)]
embed_df = pd.DataFrame(embeddings, columns=embed_cols)
embed_df["filename"] = df["filename"]
embed_df["filepath"] = df["filepath"]
embed_df["label_int"] = df["label_int"]
if "audio_batch" in df.columns:
    embed_df["audio_batch"] = df["audio_batch"]

embed_df.to_csv(WAVLM_CSV, index=False)
print(f"Saved: {WAVLM_CSV} ({len(embed_df)} rows x {len(embed_cols)} features)")

## 2. Load Features + Shared Split

In [ ]:
df_text = pd.read_csv(FEATURES_CSV)
df_text = df_text[df_text["label_int"].isin([0, 1])].reset_index(drop=True)

df_wl = pd.read_csv(WAVLM_CSV)

assert len(df_text) == len(df_wl), f"Row mismatch: text={len(df_text)}, wavlm={len(df_wl)}"

y = df_text["label_int"].values

TEXT_FEATURES = [
    "filler_rate", "filler_count", "repetition_rate", "repair_rate",
    "ttr", "mattr", "complex_word_rate", "avg_word_length",
    "n_words", "n_unique_words",
    "avg_sentence_length", "std_sentence_length", "fragment_rate", "n_sentences",
    "self_ref_rate", "discourse_marker_rate", "hedge_rate",
    "noun_rate", "verb_rate", "adj_rate",
]
PAUSE_FEATURES = [
    "pause_mean", "pause_std", "pause_median", "pause_skew",
    "long_pause_rate", "pause_ratio", "n_pauses", "pause_regularity",
    "pause_before_content_ratio", "pause_before_function_ratio",
    "mid_phrase_pause_rate", "words_per_sec", "articulation_rate",
]
PROSODIC_FEATURES = [
    "f0_mean", "f0_std", "f0_range", "f0_skew", "f0_slope",
    "energy_mean", "energy_std", "speaking_rate_std",
]
TEXT_ONLY = TEXT_FEATURES + PAUSE_FEATURES + PROSODIC_FEATURES
text_cols = [c for c in TEXT_ONLY if c in df_text.columns]
wavlm_cols = [c for c in df_wl.columns if c.startswith("wavlm_")]

print(f"Samples: {len(df_text)}")
print(f"Text features:  {len(text_cols)}")
print(f"WavLM features: {len(wavlm_cols)}")

idx_train, idx_test = train_test_split(
    np.arange(len(y)), test_size=0.2, random_state=42, stratify=y
)
y_train, y_test = y[idx_train], y[idx_test]

print(f"\nTrain: {len(y_train)} (cheating={y_train.sum()})")
print(f"Test:  {len(y_test)} (cheating={y_test.sum()})")

## 3. Train WavLM XGBoost

In [ ]:
X_wl = df_wl[wavlm_cols].fillna(0).values

wl_scaler = StandardScaler()
X_wl_train = wl_scaler.fit_transform(X_wl[idx_train])
X_wl_test = wl_scaler.transform(X_wl[idx_test])

wl_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.3,
    min_child_weight=3,
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=42,
)

wl_model.fit(
    X_wl_train, y_train,
    eval_set=[(X_wl_train, y_train), (X_wl_test, y_test)],
    verbose=20,
)

wl_proba_test = wl_model.predict_proba(X_wl_test)[:, 1]
wl_preds_test = (wl_proba_test >= 0.5).astype(int)

print(f"\n--- WavLM XGBoost (test set) ---")
print(f"Accuracy: {accuracy_score(y_test, wl_preds_test):.4f}")
print(f"F1:       {f1_score(y_test, wl_preds_test, zero_division=0):.4f}")
print(classification_report(y_test, wl_preds_test, target_names=["not cheating", "cheating"]))

## 4. Train Text XGBoost (same split)

In [ ]:
X_text = df_text[text_cols].fillna(0).values

text_scaler = StandardScaler()
X_text_train = text_scaler.fit_transform(X_text[idx_train])
X_text_test = text_scaler.transform(X_text[idx_test])

text_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=42,
)

text_model.fit(
    X_text_train, y_train,
    eval_set=[(X_text_train, y_train), (X_text_test, y_test)],
    verbose=20,
)

text_proba_test = text_model.predict_proba(X_text_test)[:, 1]
text_preds_test = (text_proba_test >= 0.5).astype(int)

print(f"\n--- Text XGBoost (test set) ---")
print(f"Accuracy: {accuracy_score(y_test, text_preds_test):.4f}")
print(f"F1:       {f1_score(y_test, text_preds_test, zero_division=0):.4f}")
print(classification_report(y_test, text_preds_test, target_names=["not cheating", "cheating"]))

## 5. Optimal Weights (2-model: WavLM + Text)

In [ ]:
print("="*60)
print("2-MODEL WEIGHT SEARCH (test set)")
print("="*60)

print(f"\nIndividual models:")
print(f"  WavLM XGBoost: F1={f1_score(y_test, wl_preds_test, zero_division=0):.4f}")
print(f"  Text XGBoost:  F1={f1_score(y_test, text_preds_test, zero_division=0):.4f}")

best_f1, best_w = 0, 0
print(f"\n  Weighted combinations (w=wavlm weight):")
for w in np.arange(0, 1.01, 0.05):
    combined = w * wl_proba_test + (1 - w) * text_proba_test
    preds = (combined >= 0.5).astype(int)
    f = f1_score(y_test, preds, zero_division=0)
    acc = accuracy_score(y_test, preds)
    marker = ""
    if f > best_f1:
        best_f1 = f
        best_w = round(w, 2)
        marker = " <-- best"
    label = "text only" if w == 0 else "wavlm only" if w == 1.0 else f"wl={w:.2f}"
    print(f"    {label:>12s}: Acc={acc:.4f}, F1={f:.4f}{marker}")

W_WL = best_w
W_TEXT = round(1 - best_w, 2)
print(f"\nBest: wavlm={W_WL}, text={W_TEXT} -> F1={best_f1:.4f}")

## 6. Threshold Sweep

In [ ]:
best_combined = W_WL * wl_proba_test + W_TEXT * text_proba_test

thresholds = np.arange(0.10, 0.91, 0.05)
rows_t = []
for t in thresholds:
    preds = (best_combined >= t).astype(int)
    rows_t.append({
        "threshold": round(t, 2),
        "precision": round(precision_score(y_test, preds, zero_division=0), 4),
        "recall": round(recall_score(y_test, preds, zero_division=0), 4),
        "f1": round(f1_score(y_test, preds, zero_division=0), 4),
        "flagged": int(preds.sum()),
        "missed": int(((y_test == 1) & (preds == 0)).sum()),
        "false_alarms": int(((y_test == 0) & (preds == 1)).sum()),
    })

thresh_df = pd.DataFrame(rows_t)
best_row = thresh_df.loc[thresh_df["f1"].idxmax()]

print(f"Threshold sweep (wavlm={W_WL}, text={W_TEXT})")
print(f"{'thresh':>7s} {'prec':>7s} {'recall':>7s} {'f1':>7s} {'flagged':>8s} {'missed':>7s} {'false_alarm':>11s}")
print("-" * 62)
for _, r in thresh_df.iterrows():
    marker = " <-- best F1" if r["threshold"] == best_row["threshold"] else ""
    print(f"  {r['threshold']:.2f}   {r['precision']:.4f}  {r['recall']:.4f}  {r['f1']:.4f}  {r['flagged']:>6d}  {r['missed']:>6d}  {r['false_alarms']:>6d}{marker}")

CHOSEN_THRESHOLD = best_row["threshold"]
print(f"\nBest F1 at threshold={CHOSEN_THRESHOLD:.2f}: F1={best_row['f1']:.4f}")

## 7. 5-Fold Cross-Validation

In [ ]:
X_wl_all = df_wl[wavlm_cols].fillna(0).values
X_text_all = df_text[text_cols].fillna(0).values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (tr_idx, te_idx) in enumerate(skf.split(X_wl_all, y), 1):
    # WavLM XGBoost
    sc_w = StandardScaler()
    X_w_tr = sc_w.fit_transform(X_wl_all[tr_idx])
    X_w_te = sc_w.transform(X_wl_all[te_idx])
    m_w = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.3,
        scale_pos_weight=(y[tr_idx]==0).sum() / max((y[tr_idx]==1).sum(), 1),
        eval_metric="logloss", random_state=42,
    )
    m_w.fit(X_w_tr, y[tr_idx], verbose=False)
    wl_prob = m_w.predict_proba(X_w_te)[:, 1]

    # Text XGBoost
    sc_t = StandardScaler()
    X_t_tr = sc_t.fit_transform(X_text_all[tr_idx])
    X_t_te = sc_t.transform(X_text_all[te_idx])
    m_t = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y[tr_idx]==0).sum() / max((y[tr_idx]==1).sum(), 1),
        eval_metric="logloss", random_state=42,
    )
    m_t.fit(X_t_tr, y[tr_idx], verbose=False)
    text_prob = m_t.predict_proba(X_t_te)[:, 1]

    combined = W_WL * wl_prob + W_TEXT * text_prob
    preds = (combined >= CHOSEN_THRESHOLD).astype(int)

    f = f1_score(y[te_idx], preds, zero_division=0)
    p = precision_score(y[te_idx], preds, zero_division=0)
    r = recall_score(y[te_idx], preds, zero_division=0)
    fold_results.append({"fold": fold, "f1": f, "precision": p, "recall": r})
    print(f"  Fold {fold}: F1={f:.4f}  Prec={p:.4f}  Rec={r:.4f}")

fold_df = pd.DataFrame(fold_results)
print(f"\n  Mean:  F1={fold_df['f1'].mean():.4f} +/- {fold_df['f1'].std():.4f}")
print(f"         Prec={fold_df['precision'].mean():.4f} +/- {fold_df['precision'].std():.4f}")
print(f"         Rec={fold_df['recall'].mean():.4f} +/- {fold_df['recall'].std():.4f}")

## 8. Save Models

In [ ]:
SAVE_DIR = "checkpoints_wavlm"
os.makedirs(SAVE_DIR, exist_ok=True)

wl_model.save_model(f"{SAVE_DIR}/xgboost_wavlm.json")
joblib.dump(wl_scaler, f"{SAVE_DIR}/scaler_wavlm.pkl")

text_model.save_model(f"{SAVE_DIR}/xgboost_text.json")
joblib.dump(text_scaler, f"{SAVE_DIR}/scaler_text.pkl")

config = {
    "text_feature_columns": text_cols,
    "wavlm_feature_columns": wavlm_cols,
    "weights": {"wavlm": W_WL, "text": W_TEXT},
    "threshold": float(CHOSEN_THRESHOLD),
    "test_f1": round(best_f1, 4),
    "cv_f1_mean": round(fold_df["f1"].mean(), 4),
    "n_train": len(idx_train),
    "n_test": len(idx_test),
}
with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Saved to {SAVE_DIR}/:")
print(f"  xgboost_wavlm.json + scaler_wavlm.pkl")
print(f"  xgboost_text.json  + scaler_text.pkl")
print(f"  results.json")

## 9. Predict on All Data + Error Analysis

In [ ]:
wl_all_proba = wl_model.predict_proba(wl_scaler.transform(X_wl_all))[:, 1]
text_all_proba = text_model.predict_proba(text_scaler.transform(X_text_all))[:, 1]
combined_all = W_WL * wl_all_proba + W_TEXT * text_all_proba

df_text["wavlm_score"] = wl_all_proba
df_text["text_score"] = text_all_proba
df_text["combined_score"] = combined_all
df_text["pred_label"] = (combined_all >= CHOSEN_THRESHOLD).astype(int)
df_text["pred_label_str"] = df_text["pred_label"].map({1: "cheating", 0: "not cheating"})

print(f"Predictions ({len(df_text)} files):")
print(f"  Cheating:     {(df_text['pred_label']==1).sum()}")
print(f"  Not cheating: {(df_text['pred_label']==0).sum()}")
print(f"  Weights: wavlm={W_WL}, text={W_TEXT}")
print(f"  Threshold: {CHOSEN_THRESHOLD:.2f}")

wrong = df_text[df_text["label_int"] != df_text["pred_label"]]
fp = wrong[wrong["pred_label"] == 1]
fn = wrong[wrong["pred_label"] == 0]
print(f"\nMisclassifications: {len(wrong)} ({len(fp)} FP, {len(fn)} FN)")

if len(wrong) > 0:
    show_cols = ["filename", "label_int", "pred_label_str", "combined_score", "wavlm_score", "text_score"]
    if "audio_batch" in wrong.columns:
        show_cols.insert(1, "audio_batch")
    show_cols = [c for c in show_cols if c in wrong.columns]
    print(wrong[show_cols].to_string(index=False))

out_cols = ["filename", "filepath", "label_int", "pred_label_str", "combined_score", "wavlm_score", "text_score"]
if "audio_batch" in df_text.columns:
    out_cols.insert(2, "audio_batch")
out_cols = [c for c in out_cols if c in df_text.columns]
df_text[out_cols].to_csv("predictions_wavlm.csv", index=False)
print(f"\nSaved: predictions_wavlm.csv")